# Customer Churn, Exploratory Data Analysis

Using the IBM Telco Customer Churn dataset: 7,043 customers, 21 features covering demographics, account details, and service subscriptions. The goal here is to get a clear picture of who churns and why before building any model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings("ignore")
os.makedirs("../visuals", exist_ok=True)

PALETTE = {"No": "#4C72B0", "Yes": "#DD8452"}
plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

print(f"Shape       : {df.shape}")
print(f"Missing vals: {df.isnull().sum().sum()}")
df.head()

Shape       : (7043, 21)
Missing vals: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Overall churn rate is around 26%, a meaningful class imbalance that I'll account for when training.

In [3]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=[PALETTE[k] for k in churn_counts.index], edgecolor="white", linewidth=1.2)
for i, (val, pct) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, val + 30, f"{val:,}\n({pct:.1f}%)", ha="center", fontsize=10)
axes[0].set_title("Churn Distribution", fontsize=13)
axes[0].set_ylabel("Customer Count")

axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=[PALETTE[k] for k in churn_counts.index],
            autopct="%1.1f%%", startangle=140, textprops={"fontsize": 11})
axes[1].set_title("Churn Share", fontsize=13)

plt.suptitle("Customer Churn Overview", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/churn_overview.png", bbox_inches="tight")
plt.show()
print(f"Overall churn rate: {churn_pct['Yes']:.2f}%")

Overall churn rate: 26.54%


Tenure and monthly charges are the most informative numerical features. Churners tend to have shorter tenure and higher monthly charges, which makes sense, newer customers on pricier plans haven't built the stickiness to stay.

In [4]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(num_cols):
    for churn_val, color in PALETTE.items():
        subset = df[df["Churn"] == churn_val][col]
        axes[0, i].hist(subset, bins=30, alpha=0.6, color=color,
                        label=f"Churn={churn_val}", edgecolor="white")
    axes[0, i].set_title(f"{col} Distribution")
    axes[0, i].legend(fontsize=8)

    df.boxplot(column=col, by="Churn", ax=axes[1, i],
               boxprops=dict(color="#333"), medianprops=dict(color="red", linewidth=2))
    axes[1, i].set_title(f"{col} by Churn")

plt.suptitle("Numerical Feature Distributions", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/numerical_distributions.png", bbox_inches="tight")
plt.show()

Contract type is the most predictive categorical feature, month-to-month customers churn at over 40% while two-year contract customers barely churn at all. Internet service and payment method also show clear separation.

In [5]:
cat_cols = ["gender", "SeniorCitizen", "Partner", "Dependents",
            "PhoneService", "InternetService", "Contract",
            "PaymentMethod", "PaperlessBilling"]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = (
        df.groupby(col)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .reset_index()
    )
    churn_rate.columns = [col, "ChurnRate"]
    churn_rate = churn_rate.sort_values("ChurnRate", ascending=False)

    bars = axes[i].bar(churn_rate[col].astype(str), churn_rate["ChurnRate"],
                       color="#4C72B0", edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, churn_rate["ChurnRate"]):
        axes[i].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5,
                     f"{val:.1f}%", ha="center", va="bottom", fontsize=8)
    axes[i].set_title(f"Churn Rate by {col}", fontsize=11)
    axes[i].set_ylabel("Churn Rate (%)")
    axes[i].tick_params(axis="x", rotation=20)
    axes[i].set_ylim(0, churn_rate["ChurnRate"].max() * 1.25)

plt.suptitle("Churn Rate by Categorical Features", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/categorical_churn_rates.png", bbox_inches="tight")
plt.show()

Tenure and TotalCharges are highly correlated (naturally, longer customers accumulate more total spend). MonthlyCharges has a positive correlation with churn while tenure is negative, confirming what the distributions showed.

In [6]:
df_enc = df.copy()
df_enc["Churn_bin"] = (df_enc["Churn"] == "Yes").astype(int)

binary_map = {"Yes": 1, "No": 0, "Male": 1, "Female": 0}
for col in ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
    df_enc[col] = df_enc[col].map(binary_map).fillna(df_enc[col])

corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen",
             "gender", "Partner", "Dependents", "PhoneService",
             "PaperlessBilling", "Churn_bin"]

corr = df_enc[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("../visuals/correlation_heatmap.png", bbox_inches="tight")
plt.show()

Summary: short tenure, high monthly charges, month-to-month contracts, and fiber optic internet are the strongest churn signals. The model in notebook 02 will formalize these patterns.

In [7]:
churn_rate = (df["Churn"] == "Yes").mean() * 100
mc_diff = df.groupby("Churn")["MonthlyCharges"].mean()
tenure_diff = df.groupby("Churn")["tenure"].mean()
contract_churn = df.groupby("Contract")["Churn"].apply(lambda x: (x == "Yes").mean() * 100)

print(f"Overall churn rate         : {churn_rate:.1f}%")
print(f"Avg monthly charge (No)    : ${mc_diff['No']:.2f}")
print(f"Avg monthly charge (Yes)   : ${mc_diff['Yes']:.2f}")
print(f"Avg tenure (No)            : {tenure_diff['No']:.1f} months")
print(f"Avg tenure (Yes)           : {tenure_diff['Yes']:.1f} months")
print(f"Month-to-month churn rate  : {contract_churn['Month-to-month']:.1f}%")
print(f"Two year churn rate        : {contract_churn['Two year']:.1f}%")

Overall churn rate         : 26.5%
Avg monthly charge (No)    : $61.27
Avg monthly charge (Yes)   : $74.44
Avg tenure (No)            : 37.6 months
Avg tenure (Yes)           : 18.0 months
Month-to-month churn rate  : 42.7%
Two year churn rate        : 2.8%
